# Silver layer

## Customers' additional information (birthdate and gender)

Table contains additional information per customer:
* Id
* Birth date
* Gender

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length, lag, date_add, lead, isnull, ifnull
from pyspark.sql.types import *

## Read table from bronze layer

In [0]:
df = spark.read.table("db_project.bronze.erp_cust_az12")
df.display()

## Check ids

In [0]:
test_id = df.select(
    min(df.CID).alias("min CID length"),
    max(df.CID).alias("max CID length"),
    F.avg(length(df.CID)).alias("avg CID length")
)
test_id.display()

Different sized ids occur, so group by id length - to see possible combinations

In [0]:
#df.select(length(df.CID)).distinct().display()
df.groupBy(length(df.CID).alias("Possible CID len")).count().withColumnRenamed("count", "Amount of CIDs").display()

There are only two types od ids:
* "AW00022042" - which is similar to key used in cust_info table
* "NASAW00022041" - seems to have additional "NAS" at the beginning, which needs to be removed

In [0]:
df = df.withColumn("CID", 
              F.when(length(df.CID) == 13, F.substring(df.CID, 4, length(df.CID)))
              .otherwise(df.CID)
)
df.display()

## Check gender

See what distinct genders are within the table

In [0]:
df.select(df.GEN).distinct().display()

There are multiple possibilities, from abbreviations to whitespaces:
* If it's F, Female - Female
* If it's M, Male - Male
* If it's whitespaces or Null - n/a\
Additionally there are records with extra spaces, which will be removed with F.trim()

In [0]:
df = df.withColumn("gen", 
              F.when(F.upper(F.trim(df.gen)).isin("FEMALE", "F"), "Female")
              .when(F.upper(F.trim(df.gen)).isin("MALE", "M"), "Male")
              .otherwise("n/a")
)
df.display()

## Check dates

In [0]:
test_dates = df.select(
    min(df.BDATE).alias("min BDATE"),
    max(df.BDATE).alias("max BDATE")
)
test_dates.display()

In [0]:
max_date = df.select(max(df.bdate)).collect()[0][0]
# display(max_date)
df.where(df.bdate <= max_date).sort(df.bdate, ascending=False).limit(50).display()

Biggest birth dates are possibly a mistake, instead they will be replaced with None.
* As all incorrect dates are over actual date - anything bigger than today's date will be replaced

In [0]:
df = df.withColumn("bdate", 
              F.when(df.bdate > current_date(), None)
              .otherwise(df.BDATE)
)
df.sort("bdate", ascending=False).display()

# Write table silver.erp_cust_az12

Everything looks ok, so save DataFrame into Delta Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", True).format("delta").saveAsTable("db_project.silver.erp_cust_az12")